In [3]:
from qiskit_ibm_runtime import QiskitRuntimeService

# Execute esta célula UMA VEZ para salvar suas credenciais
QiskitRuntimeService.save_account(
    channel="ibm_cloud",
    token="15pwb8-ZqSTNA9JbTQJX1ichqRY1D07ESrFQ7TtP-hLq",  # Cole o token que você gerou
    instance="crn:v1:bluemix:public:quantum-computing:us-east:a/c32bc959b9024b8da06b6bbc92713040:7cfb645a-ab7c-404b-ab40-5d0dc662a74a::"
)

print("Conta configurada com sucesso!")

AccountAlreadyExistsError: 'Named account (default-ibm-cloud) already exists. Set overwrite=True to overwrite.'

In [4]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
print("Backends disponíveis na sua instância xb1:")
for backend in service.backends():
    print(f"- {backend.name} ({backend.num_qubits} qubits)")

Backends disponíveis na sua instância xb1:
- ibm_fez (156 qubits)
- ibm_marrakesh (156 qubits)
- ibm_torino (133 qubits)


In [5]:
!pip install qiskit.primitives

ERROR: Could not find a version that satisfies the requirement qiskit.primitives (from versions: none)
ERROR: No matching distribution found for qiskit.primitives


In [6]:
%pip install --upgrade "qiskit>=1.0"
%pip install --upgrade "qiskit-algorithms>=0.2.0"

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [7]:
import qiskit
print(qiskit.__version__)

2.2.3


In [ ]:
pip install qiskit-aer

In [77]:
# Importações ATUALIZADAS para Qiskit 2.2.3
import numpy as np
import pandas as pd
from itertools import combinations
import matplotlib.pyplot as plt

# Qiskit imports - VERSÃO 2.2.3
from qiskit import QuantumCircuit
from qiskit_algorithms import QAOA
from qiskit_algorithms.optimizers import COBYLA
from qiskit.primitives import StatevectorSampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp

print("✅ Bibliotecas importadas com sucesso para Qiskit 2.2.3!")

✅ Bibliotecas importadas com sucesso para Qiskit 2.2.3!


In [81]:
# ===========================================================================
# CÉLULA 2: DADOS DO PROBLEMA DE PORTFÓLIO
# ===========================================================================

print("📊 CONFIGURANDO PROBLEMA DE OTIMIZAÇÃO DE PORTFÓLIO")
print("=" * 60)

N = 4  # 4 ativos para teste rápido
mu = np.array([0.12, 0.10, 0.14, 0.07])  # Retornos esperados
cov = np.array([
    [0.1, 0.02, 0.01, 0.03],
    [0.02, 0.15, 0.05, 0.02],
    [0.01, 0.05, 0.2, 0.04],
    [0.03, 0.02, 0.04, 0.1]
])
budget = 2  # Selecionar exatamente 2 ativos

print(f"• Número de ativos: {N}")
print(f"• Budget: {budget} ativos")
print(f"• Retornos esperados: {mu}")

📊 CONFIGURANDO PROBLEMA DE OTIMIZAÇÃO DE PORTFÓLIO
• Número de ativos: 4
• Budget: 2 ativos
• Retornos esperados: [0.12 0.1  0.14 0.07]


In [82]:
# ===========================================================================
# CÉLULA 3: HAMILTONIANO CORRIGIDO CIENTIFICAMENTE
# ===========================================================================

print("\n🔧 CONSTRUINDO HAMILTONIANO COM FORMULAÇÃO CORRIGIDA")
print("=" * 60)

def build_correct_hamiltonian(mu, cov, budget, penalty=5.0):  # Aumentei a penalidade para 5.0
    """
    Construção cientificamente correta do Hamiltoniano para otimização de portfólio
    """
    N = len(mu)
    
    # Formulação QUBO correta: minimizar x^T Σ x - μ^T x + P*(sum(x) - budget)^2
    Q = np.zeros((N, N))
    q = np.zeros(N)
    
    # 1. Termo de risco (variância)
    Q += cov
    
    # 2. Termo de retorno (maximização = minimizar negativo)
    for i in range(N):
        q[i] = -mu[i]
    
    # 3. Termo de penalidade para restrição de budget
    for i in range(N):
        Q[i, i] += penalty * (1 - 2 * budget)
        q[i] += 2 * penalty * budget
    
    for i in range(N):
        for j in range(i+1, N):
            Q[i, j] += 2 * penalty
            Q[j, i] += 2 * penalty
    
    constante = penalty * budget**2
    
    print("✅ QUBO construído:")
    print(f"   • Penalidade: {penalty}")
    print(f"   • Constante: {constante}")
    
    # Conversão matemática exata QUBO → Ising
    h = np.zeros(N)
    J = np.zeros((N, N))
    
    for i in range(N):
        h[i] += -0.5 * q[i]
        for j in range(N):
            if i == j:
                h[i] += -0.5 * Q[i, j]
            else:
                h[i] += -0.25 * Q[i, j]
    
    for i in range(N):
        for j in range(i+1, N):
            J[i, j] = 0.25 * Q[i, j]
    
    ising_constante = constante + 0.5 * np.sum(q) + 0.25 * np.sum(Q)
    
    # Construir operador Pauli
    pauli_terms = []
    coefficients = []
    
    pauli_terms.append("I" * N)
    coefficients.append(ising_constante)
    
    for i in range(N):
        if abs(h[i]) > 1e-10:
            pauli_str = ["I"] * N
            pauli_str[i] = "Z"
            pauli_terms.append("".join(pauli_str))
            coefficients.append(h[i])
    
    for i in range(N):
        for j in range(i+1, N):
            if abs(J[i, j]) > 1e-10:
                pauli_str = ["I"] * N
                pauli_str[i] = "Z"
                pauli_str[j] = "Z"
                pauli_terms.append("".join(pauli_str))
                coefficients.append(J[i, j])
    
    hamiltonian = SparsePauliOp(pauli_terms, coefficients)
    
    print("✅ Hamiltoniano de Ising construído:")
    print(f"   • Número de termos: {len(hamiltonian)}")
    print(f"   • Norma do Hamiltoniano: {np.linalg.norm(hamiltonian.coeffs):.4f}")
    
    return hamiltonian

# Construir Hamiltoniano com penalidade otimizada
hamiltonian = build_correct_hamiltonian(mu, cov, budget, penalty=5.0)


🔧 CONSTRUINDO HAMILTONIANO COM FORMULAÇÃO CORRIGIDA
✅ QUBO construído:
   • Penalidade: 5.0
   • Constante: 20.0
✅ Hamiltoniano de Ising construído:
   • Número de termos: 11
   • Norma do Hamiltoniano: 77.8893


In [84]:
# ===========================================================================
# CÉLULA 4: SOLUÇÃO CLÁSSICA PARA COMPARAÇÃO
# ===========================================================================

print("\n🎯 CALCULANDO SOLUÇÃO CLÁSSICA ÓTIMA")
print("=" * 60)

def classical_solution(mu, cov, budget):
    N = len(mu)
    best_value = float('inf')
    best_combination = None
    best_vector = None
    
    for comb in combinations(range(N), budget):
        x = np.zeros(N)
        x[list(comb)] = 1
        value = x @ cov @ x - mu @ x
        if value < best_value:
            best_value = value
            best_combination = comb
            best_vector = x.copy()
    
    return best_combination, best_vector, best_value

classical_comb, classical_vec, classical_val = classical_solution(mu, cov, budget)
print(f"✅ Solução clássica ÓTIMA: ativos {classical_comb}")
print(f"Valor da função objetivo: {classical_val:.6f}")



🎯 CALCULANDO SOLUÇÃO CLÁSSICA ÓTIMA
✅ Solução clássica ÓTIMA: ativos (0, 2)
Valor da função objetivo: 0.060000


In [74]:
# ===========================================================================
# CÉLULA 5: ALGORITMO QAOA OTIMIZADO
# ===========================================================================

print("\n🚀 EXECUTANDO ALGORITMO QAOA OTIMIZADO")
print("=" * 60)

try:
    # Configuração científica do QAOA
    print("🎯 CONFIGURAÇÃO DO QAOA:")
    print("• Sampler: StatevectorSampler (simulação exata)")
    print("• Otimizador: COBYLA (maxiter=100)")
    print("• Camadas QAOA: p=2 (melhor aproximação)")
    print("• Ponto inicial: [1.0, 0.5, 0.8, 0.3]")
    
    sampler = StatevectorSampler()
    optimizer = COBYLA(maxiter=100)
    
    # QAOA com mais camadas para melhor qualidade
    qaoa = QAOA(
        sampler=sampler,
        optimizer=optimizer, 
        reps=2,
        initial_point=[1.0, 0.5, 0.8, 0.3]  # 4 parâmetros para p=2
    )
    
    print("\n⚡ Executando algoritmo quântico-clássico...")
    result_local = qaoa.compute_minimum_eigenvalue(hamiltonian)

    print(f"✅ QAOA executado com sucesso!")
    print(f"Autovalor encontrado: {result_local.eigenvalue:.6f}")
    
    # Cálculo do gap
    quantum_value = result_local.eigenvalue.real
    gap_percent = abs(quantum_value - classical_val) / abs(classical_val) * 100
    print(f"Gap quântico-clássico: {gap_percent:.2f}%")
    
    # Informações dos parâmetros
    if hasattr(result_local, 'optimal_parameters'):
        print(f"Parâmetros ótimos: {result_local.optimal_parameters}")
    
    # ===========================================================================
    # ANÁLISE DETALHADA DOS RESULTADOS QUÂNTICOS
    # ===========================================================================
    
    print(f"\n🔍 ANALISANDO RESULTADOS QUÂNTICOS...")
    
    if hasattr(result_local, 'eigenstate') and result_local.eigenstate is not None:
        if isinstance(result_local.eigenstate, dict):
            # Tratamento para formato de dicionário (contagens)
            counts = result_local.eigenstate
            total_shots = sum(counts.values())
            
            print(f"📊 DISTRIBUIÇÃO DE MEDIÇÕES ({total_shots} execuções):")
            print("=" * 80)
            print(f"{'Estado':<10} {'Prob.':<12} {'Valor':<12} {'Ativos':<15} {'Validação':<10}")
            print("-" * 80)
            
            # Converter contagens em probabilidades
            probabilities = {state: count/total_shots for state, count in counts.items()}
            sorted_states = sorted(probabilities.items(), key=lambda x: x[1], reverse=True)
            
            valid_solutions = []
            for bitstring, prob in sorted_states[:10]:  # Top 10 estados
                if len(bitstring) != N:
                    continue
                    
                x = np.array([int(bit) for bit in bitstring])
                objective_value = x @ cov @ x - mu @ x
                num_assets = sum(x)
                is_valid = num_assets == budget
                assets = [i for i, val in enumerate(x) if val == 1]
                
                validity = "✅ VÁLIDO" if is_valid else "❌ INVÁLIDO"
                optimal = "🌟 ÓTIMO" if assets == list(classical_comb) else ""
                
                print(f"{bitstring:<10} {prob:<12.4f} {objective_value:<12.4f} {str(assets):<15} {validity} {optimal}")
                
                if is_valid:
                    valid_solutions.append((bitstring, objective_value, prob, assets))
            
            # Análise das soluções válidas
            if valid_solutions:
                valid_solutions.sort(key=lambda x: x[1])
                best_quantum = valid_solutions[0]
                best_gap = abs(best_quantum[1] - classical_val) / abs(classical_val) * 100
                
                print(f"\n🎯 MELHOR SOLUÇÃO QAOA ENCONTRADA:")
                print(f"• Portfólio: {best_quantum[3]}")
                print(f"• Valor objetivo: {best_quantum[1]:.6f}")
                print(f"• Probabilidade: {best_quantum[2]:.4f}")
                print(f"• Gap para ótimo: {best_gap:.2f}%")
                
                # Avaliação de qualidade
                if best_gap < 1.0:
                    quality = "🌟 EXCELENTE"
                elif best_gap < 5.0:
                    quality = "✅ MUITO BOA"
                elif best_gap < 15.0:
                    quality = "⚠️  RAZOÁVEL"
                else:
                    quality = "🔴 PRECISA MELHORAR"
                
                print(f"• Qualidade: {quality}")
                
            else:
                print("❌ Nenhuma solução válida encontrada")
                
        else:
            print("⚠️  Formato de eigenstate não suportado para análise")
    else:
        print("⚠️  Nenhum eigenstate disponível para análise")
        
except Exception as e:
    print(f"❌ ERRO no QAOA: {e}")
    import traceback
    traceback.print_exc()


🚀 EXECUTANDO ALGORITMO QAOA OTIMIZADO
🎯 CONFIGURAÇÃO DO QAOA:
• Sampler: StatevectorSampler (simulação exata)
• Otimizador: COBYLA (maxiter=100)
• Camadas QAOA: p=2 (melhor aproximação)
• Ponto inicial: [1.0, 0.5, 0.8, 0.3]

⚡ Executando algoritmo quântico-clássico...
✅ QAOA executado com sucesso!
Autovalor encontrado: 10.800371
Gap quântico-clássico: 17900.62%
Parâmetros ótimos: {ParameterVectorElement(β[0]): np.float64(2.2982612074927267), ParameterVectorElement(β[1]): np.float64(0.5321963562616938), ParameterVectorElement(γ[0]): np.float64(0.603813882264528), ParameterVectorElement(γ[1]): np.float64(0.3162023451728395)}

🔍 ANALISANDO RESULTADOS QUÂNTICOS...
📊 DISTRIBUIÇÃO DE MEDIÇÕES (1.0 execuções):
Estado     Prob.        Valor        Ativos          Validação 
--------------------------------------------------------------------------------
0000       0.4727       0.0000       []              ❌ INVÁLIDO 
1000       0.1143       -0.0200      [0]             ❌ INVÁLIDO 
0010       

In [75]:
# ===========================================================================
# CÉLULA 6: ANÁLISE COMPARATIVA AVANÇADA
# ===========================================================================

print("\n🔍 ANÁLISE COMPARATIVA CIENTÍFICA: QAOA vs CLÁSSICO")
print("=" * 80)

try:
    # Calcular todas as soluções clássicas possíveis
    all_classical_solutions = []
    for comb in combinations(range(N), budget):
        x = np.zeros(N)
        x[list(comb)] = 1
        value = x @ cov @ x - mu @ x
        all_classical_solutions.append((comb, value, x))
    
    all_classical_solutions.sort(key=lambda x: x[1])
    
    print(f"📈 ESTATÍSTICAS DAS SOLUÇÕES CLÁSSICAS:")
    print(f"• Total de combinações: {len(all_classical_solutions)}")
    print(f"• Melhor solução: {all_classical_solutions[0][1]:.6f} {all_classical_solutions[0][0]}")
    print(f"• Pior solução: {all_classical_solutions[-1][1]:.6f} {all_classical_solutions[-1][0]}")
    
    # Calcular métricas de variabilidade
    classical_values = [sol[1] for sol in all_classical_solutions]
    std_dev = np.std(classical_values)
    print(f"• Desvio padrão: {std_dev:.6f}")
    
    # ===========================================================================
    # VISUALIZAÇÃO DOS RESULTADOS
    # ===========================================================================
    
    print(f"\n📊 GERANDO VISUALIZAÇÕES...")
    
    plt.figure(figsize=(15, 10))
    
    # Subplot 1: Distribuição de soluções clássicas
    plt.subplot(2, 2, 1)
    classical_obj_values = [sol[1] for sol in all_classical_solutions]
    plt.hist(classical_obj_values, bins=10, alpha=0.7, color='blue', edgecolor='black')
    plt.axvline(classical_val, color='red', linestyle='--', linewidth=2, label=f'Ótimo: {classical_val:.4f}')
    plt.xlabel('Valor da Função Objetivo')
    plt.ylabel('Frequência')
    plt.title('Distribuição das Soluções Clássicas')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Subplot 2: Análise de risco-retorno
    plt.subplot(2, 2, 2)
    classical_returns = [mu @ sol[2] for sol in all_classical_solutions]
    classical_risks = [sol[2] @ cov @ sol[2] for sol in all_classical_solutions]
    
    plt.scatter(classical_risks, classical_returns, alpha=0.6, label='Todas soluções', s=50)
    plt.scatter([classical_risk], [classical_return], color='red', s=100, 
                label='Ótimo clássico', marker='*', edgecolor='black')
    
    # Adicionar solução QAOA se disponível
    if 'valid_solutions' in locals() and valid_solutions:
        best_quantum = valid_solutions[0]
        quantum_portfolio = np.array([int(bit) for bit in best_quantum[0]])
        quantum_return = mu @ quantum_portfolio
        quantum_risk = quantum_portfolio @ cov @ quantum_portfolio
        plt.scatter([quantum_risk], [quantum_return], color='green', s=100, 
                    label='Melhor QAOA', marker='D', edgecolor='black')
    
    plt.xlabel('Risco (Variância)')
    plt.ylabel('Retorno Esperado')
    plt.title('Fronteira Eficiente: Risco vs Retorno')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Subplot 3: Comparação de valores objetivos
    plt.subplot(2, 2, 3)
    solutions_ranking = range(1, len(all_classical_solutions) + 1)
    objective_values = [sol[1] for sol in all_classical_solutions]
    
    plt.plot(solutions_ranking, objective_values, 'bo-', alpha=0.7, label='Soluções clássicas')
    plt.axhline(y=classical_val, color='red', linestyle='--', label=f'Ótimo: {classical_val:.4f}')
    
    if 'valid_solutions' in locals() and valid_solutions:
        best_quantum_value = valid_solutions[0][1]
        plt.axhline(y=best_quantum_value, color='green', linestyle='--', 
                   label=f'Melhor QAOA: {best_quantum_value:.4f}')
    
    plt.xlabel('Ranking da Solução')
    plt.ylabel('Valor da Função Objetivo')
    plt.title('Comparação de Qualidade das Soluções')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Subplot 4: Probabilidade das soluções QAOA (se disponível)
    plt.subplot(2, 2, 4)
    if 'valid_solutions' in locals() and valid_solutions:
        valid_solutions_sorted = sorted(valid_solutions, key=lambda x: x[1])
        quantum_probs = [sol[2] for sol in valid_solutions_sorted]
        quantum_values = [sol[1] for sol in valid_solutions_sorted]
        quantum_labels = [sol[3] for sol in valid_solutions_sorted]
        
        bars = plt.bar(range(len(quantum_probs)), quantum_probs, color='green', alpha=0.7)
        plt.xlabel('Soluções QAOA (ordenadas por qualidade)')
        plt.ylabel('Probabilidade')
        plt.title('Probabilidade das Soluções Válidas QAOA')
        
        # Adicionar valores nas barras
        for i, (bar, value) in enumerate(zip(bars, quantum_values)):
            plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
                    f'{value:.4f}', ha='center', va='bottom', fontsize=8)
        
        plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

_IncompleteInputError: incomplete input (873273399.py, line 112)

In [76]:
 # ===========================================================================
    # RELATÓRIO FINAL
    # ===========================================================================
    
    print(f"\n🏆 RELATÓRIO FINAL DE DESEMPENHO")
    print("=" * 60)
    
    if 'valid_solutions' in locals() and valid_solutions:
        best_quantum = valid_solutions[0]
        best_gap = abs(best_quantum[1] - classical_val) / abs(classical_val) * 100
        total_valid_prob = sum(sol[2] for sol in valid_solutions)
        
        print(f"📊 RESULTADOS QAOA:")
        print(f"• Melhor solução: {best_quantum[3]} (valor: {best_quantum[1]:.6f})")
        print(f"• Gap para ótimo: {best_gap:.2f}%")
        print(f"• Probabilidade da melhor solução: {best_quantum[2]:.4f}")
        print(f"• Probabilidade total em soluções válidas: {total_valid_prob:.4f}")
        
        # Score de qualidade
        gap_score = max(0, 100 - best_gap)
        prob_score = best_quantum[2] * 100
        efficiency_score = total_valid_prob * 100
        
        overall_score = (gap_score + prob_score + efficiency_score) / 3
        
        print(f"\n🎯 SCORE DE QUALIDADE: {overall_score:.1f}/100")
        print(f"   • Aproximação do ótimo: {gap_score:.1f}%")
        print(f"   • Probabilidade da solução: {prob_score:.1f}%")
        print(f"   • Eficiência em soluções válidas: {efficiency_score:.1f}%")
        
        if overall_score >= 80:
            rating = "🌟 EXCELENTE"
        elif overall_score >= 60:
            rating = "✅ BOM" 
        elif overall_score >= 40:
            rating = "⚠️  RAZOÁVEL"
        else:
            rating = "🔴 PRECISA MELHORAR"
        
        print(f"   • CLASSIFICAÇÃO: {rating}")
        
except Exception as e:
    print(f"❌ Erro na análise comparativa: {e}")
    import traceback
    traceback.print_exc()

print(f"\n🎯 PRÓXIMOS PASSOS RECOMENDADOS:")
print("1. Testar com diferentes valores de penalidade (0.5, 2.0, 5.0)")
print("2. Executar com p=3 para melhorar qualidade")
print("3. Testar otimizador SPSA para melhor convergência")
print("4. Adicionar mais ativos (N=6) para teste de escalabilidade")
print("5. Executar no IBM Quantum (hardware real)")

IndentationError: unexpected indent (2540090975.py, line 5)